In [1]:
import numpy as np
from scipy.optimize import minimize_scalar, minimize, linprog, approx_fprime
from scipy.integrate import quad

## Task 1: Optimal Product Price

Profit function expanded:

$$P(x) = (x - 800)(2000 - 0.8x) = -0.8x^2 + 2640x - 1{,}600{,}000$$

**First derivative and critical point:**

$$P'(x) = -1.6x + 2640 = 0 \implies x^* = \frac{2640}{1.6} = 1650 \text{ грн}$$

**Second derivative — type of extremum:**

$$P''(x) = -1.6 < 0 \implies \text{maximum at } x^* = 1650$$

**Maximum monthly profit:**

$$P(1650) = (1650 - 800)(2000 - 0.8 \cdot 1650) = 850 \cdot 680 = 578{,}000 \text{ грн}$$

In [2]:
def profit(x):
    return (x - 800) * (2000 - 0.8 * x)

result = minimize_scalar(lambda x: -profit(x), bounds=(800, 2500), method='bounded')
x_opt = result.x
p_max = profit(x_opt)

print(f"Analytical optimum:  x* = 1650.00, P = 578000.00 грн")
print(f"Numerical optimum:   x* = {x_opt:.2f}, P = {p_max:.2f} грн")
print(f"Difference: {abs(x_opt - 1650):.6f}")

Analytical optimum:  x* = 1650.00, P = 578000.00 грн
Numerical optimum:   x* = 1650.00, P = 578000.00 грн
Difference: 0.000000


## Task 2: Minimization of a Two-Variable Function

$$f(x,y) = x^2 + xy + y^2 - 6x - 9y + 20$$

**Partial derivatives and critical point:**

$$\frac{\partial f}{\partial x} = 2x + y - 6 = 0$$
$$\frac{\partial f}{\partial y} = x + 2y - 9 = 0$$

Solving: from the first equation $y = 6 - 2x$, substituting into the second:

$$x + 2(6 - 2x) - 9 = 0 \implies -3x + 3 = 0 \implies x^* = 1,\ y^* = 4$$

**Hessian matrix:**

$$H = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$$

**Eigenvalues** via $\det(H - \lambda I) = 0$:

$$(2 - \lambda)^2 - 1 = 0 \implies \lambda_1 = 1,\quad \lambda_2 = 3$$

Both eigenvalues are positive $\implies$ $H$ is positive definite $\implies$ $(1, 4)$ is a **minimum**.

$$f(1, 4) = 1 + 4 + 16 - 6 - 36 + 20 = -1$$

In [3]:
def f(xy):
    x, y = xy
    return x**2 + x*y + y**2 - 6*x - 9*y + 20

# numerical minimum from (0,0)
res = minimize(f, x0=[0, 0], method='BFGS')
print(f"Minimum at: x = {res.x[0]:.4f}, y = {res.x[1]:.4f}, f = {res.fun:.4f}")

# stability check from three starting points
print("\nStability check:")
for x0 in [[0, 0], [10, 10], [-5, 15]]:
    r = minimize(f, x0=x0, method='BFGS')
    print(f"  Start {str(x0):<12} → x = {r.x[0]:.4f}, y = {r.x[1]:.4f}, f = {r.fun:.4f}")

Minimum at: x = 1.0000, y = 4.0000, f = -1.0000

Stability check:
  Start [0, 0]       → x = 1.0000, y = 4.0000, f = -1.0000
  Start [10, 10]     → x = 1.0000, y = 4.0000, f = -1.0000
  Start [-5, 15]     → x = 1.0000, y = 4.0000, f = -1.0000


## Task 3: Optimal Production Plan

**Variables:** $x$ — chairs, $y$ — tables

**Objective function (maximize):**
$$P(x, y) = 500x + 800y$$

**Constraints:**
$$2x + 4y \leq 120 \quad \text{(wood)}$$
$$3x + 2y \leq 90 \quad \text{(working hours)}$$
$$x \geq 0,\quad y \geq 0$$

Since `linprog` minimizes, we minimize $-P(x,y) = -500x - 800y$.

**Analytical check at corner points:** the intersection of both active constraints gives $x=15$, $y=22.5$ with $P = 25{,}500$ грн.

In [4]:
c = [-500, -800]
A_ub = [[2, 4], [3, 2]]
b_ub = [120, 90]
bounds = [(0, None), (0, None)]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
x_opt, y_opt = res.x
profit = -res.fun

print(f"Optimal plan: chairs = {x_opt:.2f}, tables = {y_opt:.2f}")
print(f"Maximum profit: {profit:.2f} грн")

wood_used = 2*x_opt + 4*y_opt
hours_used = 3*x_opt + 2*y_opt
print(f"\nResource usage:")
print(f"  Wood:         {wood_used:.2f} / 120 m²  — remainder: {120 - wood_used:.2f}")
print(f"  Working hours:{hours_used:.2f} / 90 h   — remainder: {90 - hours_used:.2f}")
print("Both resources are fully utilised — no slack.")

Optimal plan: chairs = 15.00, tables = 22.50
Maximum profit: 25500.00 грн

Resource usage:
  Wood:         120.00 / 120 m²  — remainder: 0.00
  Working hours:90.00 / 90 h   — remainder: 0.00
Both resources are fully utilised — no slack.


## Task 4: Comprehensive Production Analysis

**Part 1. Productivity dynamics:**

$$P(t) = 100 + 40t - 4t^2, \quad P'(t) = 40 - 8t$$

Critical point: $t^* = 5$ h, $P(5) = 200$ units/h

**Total output (integral):**

$$\int_0^{10}(100 + 40t - 4t^2)\,dt = \left[100t + 20t^2 - \frac{4}{3}t^3\right]_0^{10} = 1000 + 2000 - \frac{4000}{3} \approx 1666.67 \text{ units}$$

**Part 2. Initial cost parameters** from the system:

$$2x + y = 20, \quad x + 3y = 25 \implies x_0 = 7,\ y_0 = 6$$

**Cost function minimum** — partial derivatives:

$$\frac{\partial C}{\partial x} = 2x - 10 = 0 \implies x^* = 5$$
$$\frac{\partial C}{\partial y} = 2y - 8 = 0 \implies y^* = 4$$

$$C(5, 4) = 25 + 16 - 50 - 32 + 50 = 9 \text{ грн}$$

**Total production cost:**

$$\text{Total} = 1666.67 \times 9 = 15{,}000 \text{ грн}$$

In [5]:
def P(t):
    return 100 + 40*t - 4*t**2

def C(xy):
    x, y = xy
    return x**2 + y**2 - 10*x - 8*y + 50

# --- Part 1: productivity dynamics ---
print("=== Part 1: Productivity Dynamics ===")
for t in [2, 5, 8]:
    dp = approx_fprime([t], lambda x: P(x[0]), epsilon=1e-6)[0]
    trend = "growing" if dp > 0 else "at peak" if dp == 0 else "declining"
    print(f"  P'({t}) = {dp:.4f} → {trend}")

# peak productivity
peak = minimize_scalar(lambda t: -P(t), bounds=(0, 10), method='bounded')
t_star = peak.x
print(f"\nPeak at t* = {t_star:.4f} h, P(t*) = {P(t_star):.4f} units/h")

# total output
total_output, _ = quad(P, 0, 10)
print(f"Total output over shift: {total_output:.4f} units")

# --- Part 2: cost optimisation ---
print("\n=== Part 2: Cost Optimisation ===")
A_sys = np.array([[2, 1], [1, 3]])
b_sys = np.array([20, 25])
x0 = np.linalg.solve(A_sys, b_sys)
print(f"Initial parameters from system: x0 = {x0[0]:.4f}, y0 = {x0[1]:.4f}")

res_cost = minimize(C, x0=x0, method='BFGS')
x_opt, y_opt = res_cost.x
min_cost = res_cost.fun
print(f"Optimal cost parameters: x* = {x_opt:.4f}, y* = {y_opt:.4f}")
print(f"Minimum unit cost: C(x*, y*) = {min_cost:.4f} грн")

# --- Final ---
print("\n=== Final Result ===")
total_cost = total_output * min_cost
print(f"Total production cost = {total_output:.2f} × {min_cost:.2f} = {total_cost:.2f} грн")

=== Part 1: Productivity Dynamics ===
  P'(2) = 24.0000 → growing
  P'(5) = -0.0000 → declining
  P'(8) = -24.0000 → declining

Peak at t* = 5.0000 h, P(t*) = 200.0000 units/h
Total output over shift: 1666.6667 units

=== Part 2: Cost Optimisation ===
Initial parameters from system: x0 = 7.0000, y0 = 6.0000
Optimal cost parameters: x* = 5.0000, y* = 4.0000
Minimum unit cost: C(x*, y*) = 9.0000 грн

=== Final Result ===
Total production cost = 1666.67 × 9.00 = 15000.00 грн
